In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.metrics import mean_absolute_error, mean_squared_error

2026-03-07 20:21:36.029415: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Foldy podľa štúdie 
FOLD_LIST = {
    1: {"val": [4, 21, 11], "test": [8, 14, 28]},
    2: {"val": [22, 19, 28], "test": [25, 21, 1]},
    3: {"val": [4, 31, 1], "test": [34, 15, 28]},
    4: {"val": [7, 15, 13], "test": [35, 21, 1]},
    5: {"val": [7, 21, 29], "test": [35, 19, 23]},
}

def split_by_fold(df_events: pd.DataFrame, fold: int):
    spec = FOLD_LIST[fold]
    val_events = set(spec["val"])
    test_events = set(spec["test"])
    train_events = set(df_events["event_no"].unique()) - val_events - test_events

    train = df_events[df_events["event_no"].isin(train_events)].copy()
    val   = df_events[df_events["event_no"].isin(val_events)].copy()
    test  = df_events[df_events["event_no"].isin(test_events)].copy()

    return train, val, test, sorted(train_events), sorted(val_events), sorted(test_events)

In [3]:
def make_sequences_eventwise(df_part, feature_cols, target_col="DST", L=24, H=4):
    """
    Vytvorí (X, y, t) z df_part, ale vždy zvlášť v rámci event_no.
    X: (N, L, n_features), y: (N,), t: timestamp cieľa (Dst v čase t+H)
    """
    X_list, y_list, t_list, ev_list = [], [], [], []

    for ev, g in df_part.groupby("event_no"):
        g = g.sort_index()
        data = g[feature_cols + [target_col]].to_numpy(dtype=np.float32)
        times = g.index.to_numpy(dtype="datetime64[ns]")

        # index i = koniec vstupného okna (posledný vstup je i-1), cieľ je i-1+H
        # vstupné okno: [i-L, i)
        for i in range(L, len(g) - H + 1):
            x = data[i-L:i, :len(feature_cols)]
            y = data[i-1+H, len(feature_cols)]  # target_col
            t = times[i-1+H]
            X_list.append(x)
            y_list.append(y)
            t_list.append(t)
            ev_list.append(ev)

    X = np.stack(X_list) if X_list else np.empty((0, L, len(feature_cols)), dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    t = np.array(t_list)
    ev = np.array(ev_list)
    return X, y, t, ev

def fit_scalers_on_train(X_train, y_train):
    # škálujeme feature-y (na poslednej osi) a target y
    n_feat = X_train.shape[-1]
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X2 = X_train.reshape(-1, n_feat)
    x_scaler.fit(X2)

    y_scaler.fit(y_train.reshape(-1, 1))
    return x_scaler, y_scaler

def transform_Xy(X, y, x_scaler, y_scaler):
    n_feat = X.shape[-1]
    Xs = x_scaler.transform(X.reshape(-1, n_feat)).reshape(X.shape).astype(np.float32)
    ys = y_scaler.transform(y.reshape(-1, 1)).reshape(-1).astype(np.float32)
    return Xs, ys

In [4]:
def build_lstm_model(L, n_feat, hidden=64, dropout=0.2):
    inp = layers.Input(shape=(L, n_feat), name="x")
    x = layers.LSTM(hidden, return_sequences=False, name="lstm")(inp)
    x = layers.Dropout(dropout)(x)
    emb = layers.Dense(hidden, activation="relu", name="emb")(x)
    out = layers.Dense(1, name="y")(emb)

    model = Model(inp, out)
    emb_model = Model(inp, emb)  # na extrakciu embeddingu

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=[tf.keras.metrics.MeanAbsoluteError()]
    )
    return model, emb_model

In [5]:
def fit_gp_on_residuals(emb_train, y_true_s, y_pred_s):
    # reziduá v škálovanom y priestore
    r = (y_true_s - y_pred_s).reshape(-1, 1)

    # kernel: RBF + šum
    kernel = 1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-3)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0)
    gp.fit(emb_train, r.ravel())
    return gp

In [6]:
def evaluate_intervals(y_true, y_mean, y_std, alpha=0.05):
    # normálne intervaly: mean ± z * std
    # 90% => z=1.645, 95% => z=1.96
    z = 1.645 if alpha == 0.10 else 1.96
    lo = y_mean - z * y_std
    hi = y_mean + z * y_std
    coverage = np.mean((y_true >= lo) & (y_true <= hi))
    width = np.mean(hi - lo)
    return coverage, width

def train_lstm_gp_for_fold(df_events, fold=1, feature_cols=("dst","v","bz_gsm"), target_col="dst",
                           L=24, H=4, hidden=64, dropout=0.2, epochs=50, batch_size=256):
    train_df, val_df, test_df, tr_ev, va_ev, te_ev = split_by_fold(df_events, fold)
    if not isinstance(df_events.index, pd.DatetimeIndex):
        raise ValueError("df_events musí mať DatetimeIndex (nastav cez df.set_index('time1')).")
    # sequences
    Xtr, ytr, ttr, evtr = make_sequences_eventwise(train_df, list(feature_cols), target_col, L, H)
    Xva, yva, tva, evva = make_sequences_eventwise(val_df,   list(feature_cols), target_col, L, H)
    Xte, yte, tte, evte = make_sequences_eventwise(test_df,  list(feature_cols), target_col, L, H)

    if len(Xtr) == 0 or len(Xva) == 0 or len(Xte) == 0:
        raise ValueError("Niektorá split sada nemá dostatok dát na tvorbu okien. Skontroluj L/H alebo dáta.")

    # scaling (len train!)
    x_scaler, y_scaler = fit_scalers_on_train(Xtr, ytr)
    Xtr_s, ytr_s = transform_Xy(Xtr, ytr, x_scaler, y_scaler)
    Xva_s, yva_s = transform_Xy(Xva, yva, x_scaler, y_scaler)
    Xte_s, yte_s = transform_Xy(Xte, yte, x_scaler, y_scaler)

    # model
    model, emb_model = build_lstm_model(L, n_feat=Xtr_s.shape[-1], hidden=hidden, dropout=dropout)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
    ]

    model.fit(
        Xtr_s, ytr_s,
        validation_data=(Xva_s, yva_s),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # LSTM preds (scaled)
    ytr_pred_s = model.predict(Xtr_s, batch_size=1024).reshape(-1)
    yte_pred_s = model.predict(Xte_s, batch_size=1024).reshape(-1)

    # embeddings
    emb_tr = emb_model.predict(Xtr_s, batch_size=1024)
    emb_te = emb_model.predict(Xte_s, batch_size=1024)

    # GP on residuals (scaled space)
    gp = fit_gp_on_residuals(emb_tr, ytr_s, ytr_pred_s)

    # GP predict residual mean/std (scaled)
    r_mu_s, r_std_s = gp.predict(emb_te, return_std=True)
    r_std_s = r_std_s.reshape(-1)

    # combined prediction (scaled): y_hat = lstm + gp_mean_resid
    yhat_s = yte_pred_s + r_mu_s

    # prepočet späť do nT
    yhat = y_scaler.inverse_transform(yhat_s.reshape(-1,1)).reshape(-1)
    ytrue = yte
    # std v nT: keď y = a*ys + b, tak std_y = a*std_ys, kde a = y_scaler.scale_
    a = float(y_scaler.scale_[0])
    ystd = a * r_std_s  # len neistota reziduí; LSTM bodová zložka je deterministická

    # metrics
    mse = mean_squared_error(ytrue, yhat)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(ytrue, yhat)

    cov90, w90 = evaluate_intervals(ytrue, yhat, ystd, alpha=0.10)
    cov95, w95 = evaluate_intervals(ytrue, yhat, ystd, alpha=0.05)

    results = {
        "fold": fold,
        "train_events": tr_ev, "val_events": va_ev, "test_events": te_ev,
        "N_train": len(ytr), "N_val": len(yva), "N_test": len(yte),
        "RMSE_nT": rmse,
        "MAE_nT": mae,
        "PICP_90": cov90, "MPIW_90_nT": w90,
        "PICP_95": cov95, "MPIW_95_nT": w95,
    }

    artifacts = {
        "model": model,
        "emb_model": emb_model,
        "gp": gp,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "test_pred": pd.DataFrame({
            "time": pd.to_datetime(tte),
            "event_no": evte,
            "DST_true": ytrue,
            "DST_pred": yhat,
            "sigma_nT": ystd
        }).set_index("time").sort_index()
    }

    return results, artifacts

In [7]:
df_events = pd.read_csv("../../0_datasety/events_omni.csv")

df_events["time1"] = pd.to_datetime(df_events["time1"], utc=True)
df_events = df_events.set_index("time1").sort_index()
df_events["event_no"] = df_events["event_no"].astype(int)

df_events.head()

,bz_gsm,v,dst,event_no
time1,,,,
1998-04-24 05:30:00+00:00,-6.7,444.0,-56,1
1998-04-24 06:30:00+00:00,-6.9,442.0,-59,1
1998-04-24 07:30:00+00:00,-4.4,435.0,-69,1
1998-04-24 08:30:00+00:00,-1.5,437.0,-50,1
1998-04-24 09:30:00+00:00,-4.0,452.0,-38,1


In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ========== nastavenia gridu ==========
L_LIST = [6, 12, 18, 24, 30, 36, 42, 48]
H_LIST = [1, 2, 3, 4, 5, 6]
FOLD_IDS = [1, 2, 3, 4, 5]

FEATURE_COLS = ("dst", "v", "bz_gsm")
TARGET_COL = "dst"

OUT_DIR = "event_predictions_grid"
os.makedirs(OUT_DIR, exist_ok=True)

METRICS_PATH = "model_metrics.csv"
ACC_TOL_NT = 25.0

# názov súboru: event_fold{fold}_L{L_idx}_H{H}.csv
PRED_FILE_RE = re.compile(r"^event_fold(\d+)_L(\d+)_H(\d+)\.csv$")


def add_persistence(test_pred: pd.DataFrame, H: int):
    tp = test_pred.copy()
    tp["DST_persist"] = tp["DST_true"].shift(H)
    return tp


def regression_acc_within_tol(y_true, y_pred, tol=25.0):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if not np.any(m):
        return np.nan
    return float(np.mean(np.abs(y_true[m] - y_pred[m]) <= tol))


def pred_csv_path(fold, L_idx, H):
    fname = f"event_fold{fold}_L{L_idx}_H{H}.csv"
    return os.path.join(OUT_DIR, fname)


def save_metrics_df(all_metrics):
    metrics_df = (
        pd.DataFrame(all_metrics)
        .drop_duplicates(subset=["fold", "L_idx", "H"], keep="last")
        .sort_values(["fold", "L_idx", "H"])
        .reset_index(drop=True)
    )
    metrics_df.to_csv(METRICS_PATH, index=False)
    return metrics_df


def load_existing_metrics():
    if not os.path.exists(METRICS_PATH):
        return []

    df = pd.read_csv(METRICS_PATH)
    if df.empty:
        return []

    # normalizácia typov
    for c in ["fold", "L_idx", "H"]:
        if c in df.columns:
            df[c] = df[c].astype(int)

    return df.to_dict("records")


def build_completed_set_from_metrics(all_metrics):
    completed = set()
    for row in all_metrics:
        completed.add((int(row["fold"]), int(row["L_idx"]), int(row["H"])))
    return completed


def get_existing_prediction_files():
    found = {}
    for fname in os.listdir(OUT_DIR):
        m = PRED_FILE_RE.match(fname)
        if not m:
            continue
        fold, L_idx, H = map(int, m.groups())
        found[(fold, L_idx, H)] = os.path.join(OUT_DIR, fname)
    return found


def compute_metrics_from_prediction_csv(csv_path, fold, L_idx, L_value, H, acc_tol_nt):
    tp = pd.read_csv(csv_path)

    required_cols = ["DST_true", "DST_pred"]
    missing = [c for c in required_cols if c not in tp.columns]
    if missing:
        raise ValueError(f"{os.path.basename(csv_path)} nemá stĺpce: {missing}")

    y_true = tp["DST_true"].to_numpy(dtype=float)
    y_pred = tp["DST_pred"].to_numpy(dtype=float)

    row = {
        "fold": int(fold),
        "L_idx": int(L_idx),
        "L_value": int(L_value),
        "H": int(H),
        "MSE_nT2": float(mean_squared_error(y_true, y_pred)),
        "MAE_nT": float(mean_absolute_error(y_true, y_pred)),
        f"ACC_within_{int(acc_tol_nt)}nT": regression_acc_within_tol(y_true, y_pred, tol=acc_tol_nt),
        "N_test": int(np.isfinite(y_true).sum()),
    }

    # ak sú v CSV aj intervalové stĺpce, dopočítaj aj coverage štatistiky
    if {"DST_p10", "DST_p90"}.issubset(tp.columns):
        m90 = np.isfinite(tp["DST_true"]) & np.isfinite(tp["DST_p10"]) & np.isfinite(tp["DST_p90"])
        if np.any(m90):
            y = tp.loc[m90, "DST_true"].to_numpy(dtype=float)
            lo = tp.loc[m90, "DST_p10"].to_numpy(dtype=float)
            hi = tp.loc[m90, "DST_p90"].to_numpy(dtype=float)
            row["PICP_90"] = float(np.mean((y >= lo) & (y <= hi)))
            row["MPIW_90_nT"] = float(np.mean(hi - lo))
        else:
            row["PICP_90"] = np.nan
            row["MPIW_90_nT"] = np.nan
    else:
        row["PICP_90"] = np.nan
        row["MPIW_90_nT"] = np.nan

    if {"DST_p025", "DST_p975"}.issubset(tp.columns):
        m95 = np.isfinite(tp["DST_true"]) & np.isfinite(tp["DST_p025"]) & np.isfinite(tp["DST_p975"])
        if np.any(m95):
            y = tp.loc[m95, "DST_true"].to_numpy(dtype=float)
            lo = tp.loc[m95, "DST_p025"].to_numpy(dtype=float)
            hi = tp.loc[m95, "DST_p975"].to_numpy(dtype=float)
            row["PICP_95"] = float(np.mean((y >= lo) & (y <= hi)))
            row["MPIW_95_nT"] = float(np.mean(hi - lo))
        else:
            row["PICP_95"] = np.nan
            row["MPIW_95_nT"] = np.nan
    else:
        row["PICP_95"] = np.nan
        row["MPIW_95_nT"] = np.nan

    # RMSE dopočítame z MSE
    row["RMSE_nT"] = float(np.sqrt(row["MSE_nT2"]))

    return row


# ========= 1) načítaj existujúce metriky =========
all_metrics = load_existing_metrics()
completed = build_completed_set_from_metrics(all_metrics)

print(f"Načítané existujúce metriky: {len(completed)} kombinácií")

# ========= 2) nájdi existujúce predikčné CSV =========
existing_pred_files = get_existing_prediction_files()
print(f"Nájdené existujúce predikčné CSV: {len(existing_pred_files)}")

# ========= 3) dopočítaj chýbajúce metriky z existujúcich CSV (bez retréningu) =========
for fold in FOLD_IDS:
    for L_idx, L_value in enumerate(L_LIST, start=1):
        for H in H_LIST:
            key = (fold, L_idx, H)

            if key in completed:
                continue

            csv_path = existing_pred_files.get(key)
            if csv_path is None:
                continue

            try:
                row = compute_metrics_from_prediction_csv(
                    csv_path=csv_path,
                    fold=fold,
                    L_idx=L_idx,
                    L_value=L_value,
                    H=H,
                    acc_tol_nt=ACC_TOL_NT,
                )
                all_metrics.append(row)
                completed.add(key)
                save_metrics_df(all_metrics)
                print(f"Rebuilt metrics from existing CSV: fold={fold}, L_idx={L_idx}, H={H}")
            except Exception as e:
                print(f"Nepodarilo sa dopočítať metriky z {os.path.basename(csv_path)}: {e}")

# ========= 4) dotrénuj len to, čo chýba =========
for fold in FOLD_IDS:
    for L_idx, L_value in enumerate(L_LIST, start=1):
        for H in H_LIST:
            key = (fold, L_idx, H)

            if key in completed:
                print(f"SKIP: fold={fold}, L={L_value} (L_idx={L_idx}), H={H} už hotové")
                continue

            csv_path = pred_csv_path(fold, L_idx, H)

            # ak CSV existuje, ale metriky sa ešte nepodarilo urobiť vyššie, skús ešte raz z CSV
            if os.path.exists(csv_path):
                try:
                    row = compute_metrics_from_prediction_csv(
                        csv_path=csv_path,
                        fold=fold,
                        L_idx=L_idx,
                        L_value=L_value,
                        H=H,
                        acc_tol_nt=ACC_TOL_NT,
                    )
                    all_metrics.append(row)
                    completed.add(key)
                    save_metrics_df(all_metrics)
                    print(f"Recovered from CSV without retraining: fold={fold}, L_idx={L_idx}, H={H}")
                    continue
                except Exception as e:
                    print(f"CSV existuje, ale nejde použiť ({os.path.basename(csv_path)}): {e}")
                    print("Skúšam retréning...")

            print(f"Running training: fold={fold}, L={L_value} (L_idx={L_idx}), H={H}")

            try:
                res, art = train_lstm_gp_for_fold(
                    df_events,
                    fold=fold,
                    L=L_value,
                    H=H,
                    feature_cols=FEATURE_COLS,
                    target_col=TARGET_COL,
                )

                # --- predikcie pre test ---
                tp = art["test_pred"].copy()
                tp = add_persistence(tp, H=H)

                # metadáta
                tp["fold"] = fold
                tp["L_value"] = L_value
                tp["L_idx"] = L_idx
                tp["H"] = H

                # index -> stĺpec time
                tp = tp.reset_index()
                if "time" not in tp.columns:
                    tp = tp.rename(columns={tp.columns[0]: "time"})

                # --- ulož CSV ---
                tp.to_csv(csv_path, index=False)

                # --- metriky ---
                y_true = tp["DST_true"].to_numpy(dtype=float)
                y_pred = tp["DST_pred"].to_numpy(dtype=float)

                row = {
                    "fold": fold,
                    "L_idx": L_idx,
                    "L_value": L_value,
                    "H": H,
                    "MSE_nT2": float(mean_squared_error(y_true, y_pred)),
                    "MAE_nT": float(mean_absolute_error(y_true, y_pred)),
                    f"ACC_within_{int(ACC_TOL_NT)}nT": regression_acc_within_tol(
                        y_true, y_pred, tol=ACC_TOL_NT
                    ),
                    "RMSE_nT": float(res["RMSE_nT"]),
                    "PICP_90": float(res["PICP_90"]),
                    "MPIW_90_nT": float(res["MPIW_90_nT"]),
                    "PICP_95": float(res["PICP_95"]),
                    "MPIW_95_nT": float(res["MPIW_95_nT"]),
                    "N_test": int(res["N_test"]),
                }

                all_metrics.append(row)
                completed.add(key)
                save_metrics_df(all_metrics)

                print(f"Saved prediction CSV + metrics: fold={fold}, L_idx={L_idx}, H={H}")

            except Exception as e:
                print(f"ERROR pri fold={fold}, L_idx={L_idx}, H={H}: {e}")
                continue

# ========= finálny save =========
metrics_df = save_metrics_df(all_metrics)

print(f"Hotovo. Predikcie sú v: {OUT_DIR}/event_fold*_L*_H*.csv")
print(f"Súhrn metrík: {METRICS_PATH}")
display(metrics_df.tail(20))

Načítané existujúce metriky: 178 kombinácií
Nájdené existujúce predikčné CSV: 178
SKIP: fold=1, L=6 (L_idx=1), H=1 už hotové
SKIP: fold=1, L=6 (L_idx=1), H=2 už hotové
SKIP: fold=1, L=6 (L_idx=1), H=3 už hotové
SKIP: fold=1, L=6 (L_idx=1), H=4 už hotové
SKIP: fold=1, L=6 (L_idx=1), H=5 už hotové
SKIP: fold=1, L=6 (L_idx=1), H=6 už hotové
SKIP: fold=1, L=12 (L_idx=2), H=1 už hotové
SKIP: fold=1, L=12 (L_idx=2), H=2 už hotové
SKIP: fold=1, L=12 (L_idx=2), H=3 už hotové
SKIP: fold=1, L=12 (L_idx=2), H=4 už hotové
SKIP: fold=1, L=12 (L_idx=2), H=5 už hotové
SKIP: fold=1, L=12 (L_idx=2), H=6 už hotové
SKIP: fold=1, L=18 (L_idx=3), H=1 už hotové
SKIP: fold=1, L=18 (L_idx=3), H=2 už hotové
SKIP: fold=1, L=18 (L_idx=3), H=3 už hotové
SKIP: fold=1, L=18 (L_idx=3), H=4 už hotové
SKIP: fold=1, L=18 (L_idx=3), H=5 už hotové
SKIP: fold=1, L=18 (L_idx=3), H=6 už hotové
SKIP: fold=1, L=24 (L_idx=4), H=1 už hotové
SKIP: fold=1, L=24 (L_idx=4), H=2 už hotové
SKIP: fold=1, L=24 (L_idx=4), H=3 už hotové


/opt/conda/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Saved prediction CSV + metrics: fold=4, L_idx=6, H=5
Running training: fold=4, L=36 (L_idx=6), H=6
Epoch 1/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 106ms/step - loss: 0.1873 - mean_absolute_error: 0.4317 - val_loss: 0.1281 - val_mean_absolute_error: 0.3184 - learning_rate: 0.0010
Epoch 2/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 92ms/step - loss: 0.1219 - mean_absolute_error: 0.3115 - val_loss: 0.1205 - val_mean_absolute_error: 0.3055 - learning_rate: 0.0010
Epoch 3/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 98ms/step - loss: 0.1113 - mean_absolute_error: 0.2964 - val_loss: 0.1161 - val_mean_absolute_error: 0.2979 - learning_rate: 0.0010
Epoch 4/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - loss: 0.1051 - mean_absolute_error: 0.2848 - val_loss: 0.1134 - val_mean_absolute_error: 0.2950 - learning_rate: 0.0010
Epoch 5/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - loss: 0.1025 - mean_absolute_error: 0.2816 - val_loss: 0.1131 - val_mean_absolute_error: 0.2932 - learning_rate: 0.0010
Epoch 6/50
55/55 ━━━━━━━━━━━━━━━━━━